<a href="https://colab.research.google.com/github/Purvansh09/flyrannk_week1/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Purvansh09/flyrannk_week1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

This is a "which first?" ranking question (same as w04), so per the skill table I evaluate a classifier's probability, not its label, at precision@K. Starting simple: Logistic Regression first (readable, gives coefficients I can sanity-check), then Random Forest to see if nonlinearity/interactions actually earn their complexity. Both are compared against my w04 rule on the identical eligible slice, split, and metric.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
url = "https://raw.githubusercontent.com/Purvansh09/flyrannk_week1/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# rebuild the w04 baseline exactly, so it's comparable on the same rows/split later
vol_floor = df['impressions_90d'] >= 100
order = ['top_3', 'page_1', 'striking', 'page_3_5', 'deep']
sigB = (df[vol_floor & df['position_tier'].isin(order)].groupby('position_tier')
        .agg(median_ctr=('ctr', 'median')).reindex(order))
benchmark = sigB['median_ctr'].to_dict()
df['expected_ctr'] = df['position_tier'].map(benchmark)
df['ctr_gap_score'] = ((df['expected_ctr'] - df['ctr']) / df['expected_ctr']).clip(lower=0, upper=1)
df['staleness_score'] = (df['days_since_last_update'] / 365).clip(upper=1)
eligible = df['position_tier'].isin(order) & vol_floor
df['baseline_action_score'] = 0.0
df.loc[eligible, 'baseline_action_score'] = (
    50*df.loc[eligible,'staleness_score'] + 50*df.loc[eligible,'ctr_gap_score']).round(1)

work = df[eligible].copy().reset_index(drop=True)
print("Universe (same eligible slice as w04):", len(work), "rows")

# --- leakage check: trend_pct is COMPUTED from these columns, not just correlated ---
calc = (df['impressions_last_30d']-df['impressions_prev_30d'])/df['impressions_prev_30d'].replace(0,np.nan)*100
print(f"trend_pct corr with (impressions_last_30d vs prev_30d) pct change: {df['trend_pct'].corr(calc):.6f}")
print("-> confirms trend_direction/trend_pct are DERIVED from *_last_30d and *_prev_30d.")
print("-> dropping trend_direction, trend_pct, and all six *_last_30d/*_prev_30d columns from features.")


Universe (same eligible slice as w04): 22006 rows
trend_pct corr with (impressions_last_30d vs prev_30d) pct change: 1.000000
-> confirms trend_direction/trend_pct are DERIVED from *_last_30d and *_prev_30d.
-> dropping trend_direction, trend_pct, and all six *_last_30d/*_prev_30d columns from features.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Grouped by client_id, not random row-level, and not time-based (this dataset is a single 90-day snapshot, no separate time axis to split on). w04 already showed pages from the same client can share an identical update timestamp — a random split would put near-duplicate rows from the same client on both sides of train/test, which is leakage in disguise. GroupShuffleSplit guarantees zero client overlap.

Caveat worth stating plainly: there are only 30 unique clients total. An 80/20 group split puts 24 clients in train and 6 in test — small enough that which 6 clients land in test can swing the numbers. I'm reporting this split as-is rather than cherry-picking a seed that looks better.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

leak_cols = ['trend_direction','trend_pct','impressions_last_30d','clicks_last_30d','sessions_last_30d',
             'impressions_prev_30d','clicks_prev_30d','sessions_prev_30d']

numeric_features = ['search_volume','competition','cpc','word_count','char_count','content_age_days',
                     'age_tier_order','days_since_last_update','ctr','avg_position','engagement_rate',
                     'scroll_rate','ai_traffic_pct','impressions_90d','clicks_90d','pageviews_90d',
                     'sessions_90d','users_90d','engaged_sessions_90d','ai_sessions_90d',
                     'scroll_events_90d','days_with_impressions','days_with_sessions']
categorical_features = ['content_type','main_intent','age_tier','freshness_tier','word_count_tier',
                         'char_count_tier','impression_tier','position_tier','competition_level']
assert not (set(numeric_features+categorical_features) & set(leak_cols))

X = work[numeric_features + categorical_features]
y = work['is_declining_label']
groups = work['client_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

tr_c, te_c = set(work.loc[train_idx,'client_id']), set(work.loc[test_idx,'client_id'])
print(f"{len(X_train)} train / {len(X_test)} test rows | {len(tr_c)} train clients / {len(te_c)} test clients | overlap: {len(tr_c & te_c)}")

18392 train / 3614 test rows | 24 train clients / 6 test clients | overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Same test split, same metric (precision@K), same K's as w04. Base rate on the test slice: 0.553.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

preprocess_lr = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric_features),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)])
preprocess_rf = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), numeric_features),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)])

lr = Pipeline([('prep', preprocess_lr), ('clf', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))])
rf = Pipeline([('prep', preprocess_rf), ('clf', RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=RANDOM_STATE, n_jobs=-1))])
lr.fit(X_train, y_train); rf.fit(X_train, y_train)

proba_lr = lr.predict_proba(X_test)[:,1]
proba_rf = rf.predict_proba(X_test)[:,1]
baseline_test_score = work.loc[test_idx, 'baseline_action_score'].values
y_test_arr = y_test.values

def precision_at_k(scores, labels, k):
    order_idx = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order_idx[:k]].mean()

rows = []
for k in (20,50,100):
    rows.append({'K':k, 'base_rate':round(y_test_arr.mean(),3),
                 'baseline_rule_w04':round(precision_at_k(baseline_test_score,y_test_arr,k),3),
                 'logistic_regression':round(precision_at_k(proba_lr,y_test_arr,k),3),
                 'random_forest':round(precision_at_k(proba_rf,y_test_arr,k),3)})
comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))

  K  base_rate  baseline_rule_w04  logistic_regression  random_forest
 20      0.553               0.75                 0.90           0.70
 50      0.553               0.74                 0.70           0.58
100      0.553               0.69                 0.72           0.61


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

What RF leans on (permutation importance, not raw .feature_importances_, which overstates high-cardinality columns): avg_position, ctr, clicks_90d, position_tier top the list — the same signals my w04 audit already validated. Nothing is suspiciously dominant (top importance is 0.008, not near-1.0), which is itself a mild reassurance against leakage.

What LR leans on: freshness_tier_0-30 has the single largest coefficient, and its sign says fresher pages are more likely to be flagged declining — which looks backwards next to w04's aggregate finding (staler pages declined more often). This is very likely a collinearity artifact of one-hot dummy coding (each tier's coefficient is relative to the reference category, not a clean marginal effect) rather than a real reversal — a reason to trust RF's permutation importance over raw LR coefficients when reading "what matters."

Where it's wrong: 6 of RF's top-20 test picks aren't actually declining. All 6 come from just 3 clients, and every one of them has ctr=0.0, near-identical position/staleness values to true declining pages from the same client. Three concrete cases:

content_2dbab51b83c9 (client_d029fa3a95): page_1, ctr=0.00, 20 days since update — identical profile to 8 correctly-flagged rows from the same client, but this one is trending up. The model has no signal that distinguishes it from its true-positive siblings.
content_a53d1c4231b3 (same client): striking tier, ctr=0.00, same 20-day staleness — trending stable, not down.
content_500bd3907331 (client_4e07408562): page_1, ctr=0.10 (nonzero, closest to a real miss), 104 days stale — trending stable.

This is the same bulk-timestamp pattern w04's weak-picks flagged: when a whole client batch shares one update date and near-zero CTR, the model (like the rule) genuinely can't tell which pages in that batch are declining and which aren't — the input features don't carry that distinction. That's a real ceiling on this feature set, not a fixable bug.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

result = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1, scoring='roc_auc')
imp_df = pd.DataFrame({'feature': X_test.columns, 'importance': result.importances_mean}).sort_values('importance', ascending=False)
print(imp_df.head(10).to_string(index=False))

ohe_cols = lr.named_steps['prep'].named_transformers_['cat'].named_steps['ohe'].get_feature_names_out(categorical_features)
coef_df = pd.DataFrame({'feature': numeric_features + list(ohe_cols), 'coef': lr.named_steps['clf'].coef_[0]})
print(coef_df.assign(abs_coef=lambda d: d['coef'].abs()).sort_values('abs_coef', ascending=False).head(10)[['feature','coef']].to_string(index=False))

test_view = work.loc[test_idx, ['content_id','client_id','position_tier','ctr','days_since_last_update','trend_direction']].copy()
test_view['rf_proba'] = proba_rf
test_view['actual_declining'] = y_test.values
test_view = test_view.sort_values('rf_proba', ascending=False).reset_index(drop=True)
wrong = test_view.head(20)[test_view.head(20)['actual_declining']==0]
print(f"\nWrong picks in RF top 20: {len(wrong)}")
print(wrong.to_string(index=False))

              feature  importance
         avg_position    0.008022
                  ctr    0.007739
           clicks_90d    0.006643
        position_tier    0.006191
   days_with_sessions    0.004551
            users_90d    0.003547
         sessions_90d    0.003161
        pageviews_90d    0.002435
days_with_impressions    0.002279
 engaged_sessions_90d    0.001706
                  feature      coef
      freshness_tier_0-30  1.090747
                users_90d -0.931113
             sessions_90d  0.803106
    word_count_tier_<1000  0.721226
word_count_tier_1000-2000  0.694849
word_count_tier_2000-3500 -0.691689
    word_count_tier_3500+ -0.649909
   days_since_last_update  0.574396
   char_count_tier_25000+  0.504273
             avg_position -0.502172

Wrong picks in RF top 20: 6
          content_id         client_id position_tier  ctr  days_since_last_update trend_direction  rf_proba  actual_declining
content_2dbab51b83c9 client_d029fa3a95        page_1  0.0                  

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.